In [5]:
# 下载Deepresearch-9k 数据集作为通用训练数据


from datasets import load_dataset
import os

# 1. 指定下载与缓存位置
# 使用 cache_dir 参数可以将数据下载到你指定的硬盘路径，避免占用系统盘空间
custom_cache_dir = "./my_dataset_cache" 
os.makedirs(custom_cache_dir, exist_ok=True)

print("正在加载数据集并下载至:", custom_cache_dir)
ds = load_dataset("artillerywu/DeepResearch-9K", cache_dir=custom_cache_dir)

# 2. 指定下载/保留的字段
# 根据论文 [cite: 90]，该数据集包含 'question', 'difficulty', 'search_trajectory', 'final_answer'
# 我们使用 remove_columns 方法舍弃不需要的字段
cols_to_remove = ["search trajectory"]
ds_filtered = ds.remove_columns(cols_to_remove)

# 3. 转为 CSV 保存
# 因为数据集通常包含多个划分（如 train, test），建议遍历保存
for split, dataset in ds_filtered.items():
    output_name = f"deep_research_9k_{split}.csv"
    dataset.to_csv(output_name, index=False) # index=False 不保存行索引
    print(f"已将 {split} 部分保存至: {output_name}")

正在加载数据集并下载至: ./my_dataset_cache


Creating CSV from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

已将 train 部分保存至: deep_research_9k_train.csv


In [6]:
# 分析 DeepResearch-hard

import pandas as pd

df = pd.read_csv("deep_research_hard.csv")

print(df.difficulty.value_counts())
print(df.difficulty.value_counts(normalize=True))

difficulty
3    2288
2     860
1     826
Name: count, dtype: int64
difficulty
3    0.575742
2    0.216407
1    0.207851
Name: proportion, dtype: float64


In [14]:
# 从 DeepResearch-9k 的 hard 数据集 抽取出 SFT+RL 实际使用的数据集

import pandas as pd
from pathlib import Path

# 加载完整数据集
df = pd.read_csv("deep_research_hard.csv")

# 打印出当前的列名，帮你排查到底叫什么
print("📌 原始数据包含的列名有：", df.columns.tolist())

# 按照 difficulty 字段进行分层抽样，总数取 1000
sampled_df = df.groupby('difficulty', group_keys=False).sample(frac=0.25163563160544, random_state=42)

print("📌 sampled_df数据包含的列名有：", sampled_df.columns.tolist())

# 将抽样后的数据分为两部分：
# 1. 潜力 SFT 库 (用于跑 Agent 框架获取轨迹)
sft_candidates = sampled_df.sample(n=600, random_state=42) 
# 2. 预留 GRPO 验证/训练库 (不运行 Agent，仅保留问答对)
grpo_pool = sampled_df.drop(sft_candidates.index)

if 'difficulty' not in sft_candidates.columns:
    # 尝试将 index 重置为普通列
    sft_candidates = sft_candidates.reset_index()
    grpo_pool = grpo_pool.reset_index()

# 打印出当前的列名，帮你排查到底叫什么
print("📌 当前数据包含的列名有：", sft_candidates.columns.tolist())

columns_to_keep = ['question', 'difficulty', 'final answer'] 

# 过滤数据框，只保留指定列
sft_final = sft_candidates[columns_to_keep]
grpo_final = grpo_pool[columns_to_keep]


# 保存数据
output_dir = Path("general_dataset")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. 保存为 CSV 
# 增加 encoding="utf-8-sig"，防止在 Windows Excel 中打开时中文乱码
sft_final.to_csv(output_dir / "sft_candidates.csv", index=False, encoding="utf-8-sig")
grpo_final.to_csv(output_dir / "grpo_pool.csv", index=False, encoding="utf-8-sig")

# 3. 保存为 JSON
# sft_candidates.to_json(output_dir / "sft_candidates.json", orient="records", force_ascii=False, indent=2)
# grpo_pool.to_json(output_dir / "grpo_pool.json", orient="records", force_ascii=False, indent=2)

# 【进阶推荐】：对于 LLM 训练数据 (SFT/RL)，业界通常更推荐使用 JSONL (JSON Lines) 格式
# 如果你的 Agent 框架或训练框架支持，建议使用以下代码替代上面的 JSON 保存：
sft_final.to_json(output_dir / "sft_candidates.jsonl", orient="records", lines=True, force_ascii=False)
grpo_final.to_json(output_dir / "grpo_pool.jsonl", orient="records", lines=True, force_ascii=False)

print(f"✅ 数据成功保存至 {output_dir.absolute()} 目录下！")

📌 原始数据包含的列名有： ['question', 'difficulty', 'final answer']
📌 sampled_df数据包含的列名有： ['question', 'difficulty', 'final answer']
📌 当前数据包含的列名有： ['question', 'difficulty', 'final answer']
✅ 数据成功保存至 D:\DissertationProject\DissertationProject\dataset\general_dataset 目录下！


In [21]:
import json

with open("D:\\DissertationProject\\DissertationProject\SFT\\trajectories\\20260312_225354_sft_traj_task11.jsonl", "r", encoding="utf-8") as f:
    n = 0
    data = [json.loads(line) for line in f]

for item in data:
    for message in item["messages"]:
        if message['role'] == "user":
            user_input = message['content']
        if message['role'] == "assistant":
            assistant_output = message['content']
    print(user_input)
    print(assistant_output)
    break

你是一个深度研究智能体。你需要通过“思考→报告→行动”(Think-Report-Action) 的迭代循环来解决复杂问题。
你没有历史对话记录！你唯一的知识载体和上下文是“全局演进报告 (report)”。你必须在每一次迭代中，将新线索与旧报告进行深度语义融合、去重和提炼，而非简单堆砌。

**输出结构要求 (必须且只能包含以下三个 XML 标签，禁止输出任何标签外的文本)**：
1. <think> 认知暂存区：分析当前工作区状态，评估上次行动的结果(observation)，准备更新报告 (report)并推理下一步工具调用策略(action)。此部分属于你的内部推理，不会带入下一轮。</think>
2. <report> 演进报告 (中心记忆)：这是你唯一的上下文记忆！请将本次新发现与上一轮的旧报告进行综合，输出一份结构化、高信息密度的最新进展摘要。禁止简单追加，必须进行语义融合。</report>
3. <action> 行动：仅使用提供的工具，格式为"工具名[参数]"。如果认为已有足够证据回答初始问题，请输出"finish[最终回复]"，这是你向用户展示结果的唯一渠道。</action>

支持工具：
- list_skills：获取所有可用的技能（Skills）列表及其功能描述。参数格式为 {}
- get_skill_prompt：获取某一具体技能的详细 Prompt（工作流程提示）。参数格式为 {'skill_name': '需要获取的技能名称'}
- jina_reader：Read and extract content from web page.
参数格式: {"type": "object", "properties": {"url": {"type": "string", "format": "uri", "description": "URL of the webpage to read and extract content from"}, "customTimeout": {"type": "number", "description": "Override timeout in seconds for slow sites"}, "page": {"type": "number", "default": 1, "descrip